In [ ]:
# Step-1 Fetch Charged Event from CT

import requests
import json
from openpyxl import Workbook
from openpyxl.utils.exceptions import IllegalCharacterError # Import the specific exception
from google.colab import files
import re # Import the re module

# ---- CONFIG ----
ACCOUNT_ID = "6K5-56W-8K7Z"
PASSCODE = "WHO-JIE-CLEL"
BASE_URL = "https://in1.api.clevertap.com/1/events.json"
OUTPUT_FILE = "Charged from Oct'25.xlsx"

# ---- HARDCODED VALUES ----
EVENT_NAME = "Charged" # Replace with your actual event name
FROM_DATE = "20260201" # Format: YYYYMMDD
TO_DATE = "20260430"
REQUIRED_FIELDS = [] # Set to [] to disable filtering

# Regex to find illegal characters
ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')


# ---- UTILS ----
def flatten_dict(d, parent_key='', sep='.'):
    """
    Recursively flattens a nested dictionary.
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def clean_string(text):
    """
    Removes illegal characters from a string.
    """
    if isinstance(text, str):
        return ILLEGAL_CHARACTERS_RE.sub('', text)
    return text

def format_identity(identity_obj):
    """
    Helper function to format the identity field by joining all values into a single string.
    """
    if isinstance(identity_obj, dict) and identity_obj is not None:
        return ", ".join(str(v) for v in identity_obj.values())
    return identity_obj or ""

# ---- MAIN FUNCTION ----
def fetch_events(event_name, from_date, to_date):
    """
    Fetches events from the CleverTap API and writes them to an Excel file.
    """
    print(f"[INFO] Fetching events: {event_name}, From: {from_date}, To: {to_date}")
    headers = {
        "X-CleverTap-Account-Id": ACCOUNT_ID,
        "X-CleverTap-Passcode": PASSCODE,
        "Content-Type": "application/json"
    }
    payload = {
        "event_name": event_name,
        "from": int(from_date),
        "to": int(to_date),
        "event_properties": []
    }

    try:
        response = requests.post(f"{BASE_URL}?events=false", headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
    except Exception as e:
        print(f"[ERROR] Failed to fetch initial cursor: {e}")
        return

    print(f"[INFO] Initial API status: {result.get('status')}")
    cursor = result.get("cursor")

    if not cursor:
        print("[ERROR] No cursor returned.")
        print("[DEBUG] Response:", result)
        return

    all_records = []
    fetch_cursor(cursor, event_name, headers, all_records)

    if all_records:
        write_to_excel(all_records, event_name)
    else:
        print("[WARN] No records fetched.")

def fetch_cursor(cursor, event_name, headers, all_records):
    """
    Recursively fetches data using a cursor and appends records.
    """
    while cursor:
        print(f"[INFO] Fetching cursor: {cursor}")
        try:
            response = requests.get(f"{BASE_URL}?cursor={cursor}", headers=headers)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"[ERROR] Failed to fetch data for cursor {cursor}: {e}")
            break

        records = data.get("records", [])
        print(f"[INFO] Records fetched in this page: {len(records)}")

        for record in records:
            # Pre-process the identity field to handle dictionary values
            if 'profile' in record and 'identity' in record['profile']:
                record['profile']['identity'] = format_identity(record['profile']['identity'])

            # Flatten the entire record
            flat_record = flatten_dict(record)
            all_records.append(flat_record)

        cursor = data.get("next_cursor")
        if not cursor:
            print("[INFO] No more pages left.")

def write_to_excel(records, event_name):
    """
    Writes the fetched records to an Excel file.
    """
    print(f"[INFO] Preparing to write {len(records)} rows to {OUTPUT_FILE}")
    wb = Workbook()
    ws = wb.active
    ws.title = event_name[:20]

    # Step 1: Determine headers
    headers = set()
    for record in records:
        headers.update(record.keys())
    headers = sorted(headers)
    ws.append(headers)

    # Step 2: Write filtered rows
    valid_count = 0
    for record in records:
        skip = False
        for field in REQUIRED_FIELDS:
            field_value = record.get(field, "")
            if isinstance(field_value, str):
                field_value = field_value.strip().lower()
            if not field_value or field_value == "na":
                skip = True
                break
        if skip:
            continue

        row = []
        for h in headers:
            value = record.get(h, "")

            # Ensure the value is a string before cleaning and appending.
            if isinstance(value, (list, dict)):
                value = json.dumps(value)
            if value is None:
                value = ""

            # Clean the string value before appending to the row
            value = clean_string(str(value))
            row.append(value)
        ws.append(row)
        valid_count += 1

    wb.save(OUTPUT_FILE)
    print(f"[SUCCESS] File saved: {OUTPUT_FILE} with {valid_count} valid rows")


# ---- RUN THE FETCH ----
fetch_events(EVENT_NAME, FROM_DATE, TO_DATE)
